## ANA - Main Analysis of Ectoderm/Myeloid Cell Movements

### Notes

#### Information on input

- _Per-track main data_ (`Ectoderm-data_clean_vec.pkl`, `Myeloid-data_clean_vec.pkl`)
    - `ecto_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
    - `myel_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
    - `*` Details on df columns:
        - `f, t, y, x`: frames `[1]`, times `[min]`, positions `[microns]`
        - `cen_y, cen_x, y_rel, x_rel, r, clust_r`: cluster center, relative positions, radial distances, cluster size `[microns]`
        - `vy, vx, vy_itp, vx_itp`: velocities, locally interpolated ectoderm velocities `[microns/min]`
        - `v_mag, v_itp_mag`: velocity vector magnitudes
        - `vy_n, vx_n, vy_itp_n, vx_itp_n`: components of normalized (magnitude 1.0) vectors
        - `vy_traj, vx_traj, vy_traj_n, vx_traj_n`: cell track trajectory-aligned (optionally normalized) vectors
        - `vy_ray, vx_ray, vy_ray_n, vx_ray_n`: cluster radial axis-aligned (optionally normalized) vectors

* _Per-track similarity metrics_ (`Ectoderm-similarities.pkl`, `Myeloid-similarities.pkl`)
    - `ecto_similarities[conditions][positions]`; pandas dfs of shape `tracks X (f, ...)*`
    - `myel_similarities[conditions][positions]`; pandas dfs of shape `tracks X (f, ...)*`
    - `*` Details on df columns:
        - `f, t, y, x, cen_y, cen_x, y_rel, x_rel, r, clust_r`: as in `ecto_data` and `myel_data`
        - `metric + "-shift="+str(s) for s in profile_time_shifts`: Per-cell local similarity for each similarity metric

- _Time-shift profile data per frame_ (`Ectoderm-profiles.pkl`, `Myeloid-profiles.pkl`)
    - `ecto_profiles[conditions][positions][metric]`; pandas dfs of shape `f X profile_time_shifts`
    - `myel_profiles[conditions][positions][metric]`; pandas dfs of shape `f X profile_time_shifts`

* _Time-shift profile data averaged over frames_ (`Ectoderm-profiles_mean.pkl`, `Myeloid-profiles_mean.pkl`)
    - `ecto_profs_mean[conditions][positions][metric]`; pandas dfs of shape `profile_time_shifts`
    - `myel_profs_mean[conditions][positions][metric]`; pandas dfs of shape `profile_time_shifts`

- Conditions: `norm` means normal fibronectin, `dilute` / `dilu` means low fibronectin

* Tracking was done with StarDist for the ectoderm and manually in ImageJ for myeloid cells


#### Pre-requisites

- The data must have been preprocessed with `RUN - 1 - Preprocessing.ipynb`
- Movement vectors must have been extracted and interpolated with `RUN - 2 - Movement vectors.ipynb`
- Shifted time profiles must have been computed with `RUN - 3 - Correlations and similarities.ipynb`


#### Content of this notebook

1. Set parameters and load the data
2. Interactive data visualizations from different pipeline steps
3. Remove problematic outliers identified as unhealthy explants (not applicable in vivo)
4. Analysis of correlation/similarity profiles over time shifts
5. Analysis of radial velocity profiles & comparison to model
6. Ectoderm-myeloid transfer functions for cue saturation model

### Prep

In [ ]:
### Imports

%load_ext autoreload
%autoreload 2

import os, warnings, pickle
from IPython.display import display

import itertools
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors as mplcolors
import ipywidgets
from ipywidgets import interact

from scipy.spatial import distance as spdist
from scipy import stats
from scipy.optimize import curve_fit

import sys; sys.path.insert(0, '..')
from tracking_analysis.utilities import savebutton, get_shifted_data
import tracking_analysis.pcl_tools as pcl_tools

In [ ]:
### Seeding

np.random.seed(42)

In [ ]:
### Parameters

# Units
pxl_res  = 1.5152  # [microns]
time_res = 5       # [min]

# Imported: shifted time profiles
prof_min_shift  = -30              # Time point range of profile into past
prof_max_shift  =  30              # Time point ange of profile into future
prof_mean_range = pmr = (80, 180)  # Time point range for averaged profile

# Radial range of interest for clean profiles
radial_profile_range = rpr = (250, 650)

# Whether to flip profile shift sign
# Note: Initially, the corr/sim profiles were thought of as being computed "against the ectoderm as a reference", 
#       so negative shifts meaning comparisons to past ectoderm (and vice versa) made good sense. However, after 
#       the myeloid's negative-shifted peak was found, the point was made that it would make more sense to flip 
#       the shifts, thinking of the myel peak as "occurring after" the ecto peak along a left-to-right axis in 
#       time. This is implemented if below flag is set to true.
flip_shifts = True

# Condition labels dictionary
cond_dict = {'norm' : 'normal fibronectin', 'dilu' : 'diluted fibronectin'}

# Optimized y-ranges for different metrics
yranges_tight = {
    "corr"              : ( 0.00, 0.85),
    "corr_n"            : ( 0.00, 0.80),
    "speed_corr"        : (-0.15, 0.75),
    "traj_para_corr"    : (-0.20, 0.85),
    "traj_para_n_corr"  : (-0.20, 0.85),
    "traj_ortho_corr"   : (-0.20, 0.85),
    "traj_ortho_n_corr" : (-0.20, 0.85),
    "ray_para_corr"     : (-0.20, 0.90),
    "ray_para_n_corr"   : (-0.20, 0.90),
    "ray_ortho_corr"    : (-0.10, 0.85),
    "ray_ortho_n_corr"  : (-0.10, 0.85),
    "cos_sim"           : ( 0.10, 0.85),
    "speed_sim"         : ( 0.60, 0.85),
    "traj_para_sim"     : ( 0.60, 0.87),
    "traj_para_n_sim"   : ( 0.60, 0.87),
    "traj_ortho_sim"    : ( 0.60, 0.87),
    "traj_ortho_n_sim"  : ( 0.60, 0.87),
    "ray_para_sim"      : ( 0.60, 0.90),
    "ray_para_n_sim"    : ( 0.60, 0.90),
    "ray_ortho_sim"     : ( 0.60, 0.87),
    "ray_ortho_n_sim"   : ( 0.60, 0.87),
}

yranges_limits = {
    "corr"              : (-1.0, 1.0),
    "corr_n"            : (-1.0, 1.0),
    "speed_corr"        : (-1.0, 1.0),
    "traj_para_corr"    : (-1.0, 1.0),
    "traj_para_n_corr"  : (-1.0, 1.0),
    "traj_ortho_corr"   : (-1.0, 1.0),
    "traj_ortho_n_corr" : (-1.0, 1.0),
    "ray_para_corr"     : (-1.0, 1.0),
    "ray_para_n_corr"   : (-1.0, 1.0),
    "ray_ortho_corr"    : (-1.0, 1.0),
    "ray_ortho_n_corr"  : (-1.0, 1.0),
    "cos_sim"           : (-1.0, 1.0),
    "speed_sim"         : ( 0.0, 1.0),
    "traj_para_sim"     : ( 0.0, 1.0),
    "traj_para_n_sim"   : ( 0.0, 1.0),
    "traj_ortho_sim"    : ( 0.0, 1.0),
    "traj_ortho_n_sim"  : ( 0.0, 1.0),
    "ray_para_sim"      : ( 0.0, 1.0),
    "ray_para_n_sim"    : ( 0.0, 1.0),
    "ray_ortho_sim"     : ( 0.0, 1.0),
    "ray_ortho_n_sim"   : ( 0.0, 1.0),
}

In [ ]:
### Data locations

top_path = r"..\Data\ex_vivo"

ecto_base = r"Ectoderm"
myel_base = r"Myeloid"

In [ ]:
### Load the data

# Ectoderm
with open(os.path.join(top_path, ecto_base+'-data_clean_vec_prof.pkl'), "rb") as infile:
    ecto_data = pickle.load(infile)
with open(os.path.join(top_path, ecto_base+"-similarities.pkl"), "rb") as infile:
    ecto_similarities = pickle.load(infile)
with open(os.path.join(top_path, ecto_base+'-profiles.pkl'), "rb") as infile:
    ecto_profiles = pickle.load(infile)
with open(os.path.join(top_path, ecto_base+'-profiles_mean.pkl'), "rb") as infile:
    ecto_profs_mean = pickle.load(infile)

# Myeloid
with open(os.path.join(top_path, myel_base+'-data_clean_vec_prof.pkl'), "rb") as infile:
    myel_data = pickle.load(infile)
with open(os.path.join(top_path, myel_base+"-similarities.pkl"), "rb") as infile:
    myel_similarities = pickle.load(infile)
with open(os.path.join(top_path, myel_base+'-profiles.pkl'), "rb") as infile:
    myel_profiles = pickle.load(infile)
with open(os.path.join(top_path, myel_base+'-profiles_mean.pkl'), "rb") as infile:
    myel_profs_mean = pickle.load(infile)

# Report
print("\nEctoderm data:\n")
print(' ', ecto_data['norm'].keys())
print(' ', ecto_data['dilu'].keys())
print(' ', ecto_data['norm']['Pos001'].shape)
print(' ', ecto_profiles['norm']['Pos001'].keys())
print(' ', ecto_profiles['norm']['Pos001']['cos_sim'].shape)
print(' ', ecto_profs_mean['norm']['Pos001']['cos_sim'].shape)

print("\nMyeloid data:\n")
print(' ', myel_data['norm'].keys())
print(' ', myel_data['dilu'].keys())
print(' ', myel_data['norm']['Pos001'].shape)
print(' ', myel_profiles['norm']['Pos001'].keys())
print(' ', myel_profiles['norm']['Pos001']['cos_sim'].shape)
print(' ', myel_profs_mean['norm']['Pos001']['cos_sim'].shape)

In [ ]:
### Extract some useful bits and pieces

# Lists of available metrics
metrics = list(ecto_profs_mean['norm']['Pos001'].keys())
metrics_similarity = [m for m in metrics if m.endswith("_sim")]
metrics_correlation = [m for m in metrics if (m.startswith("corr") or m.endswith("_corr"))]
print("Available metrics:", metrics)
print("\n...of which are (local) similarity metrics:", metrics_similarity)
print("\n...of which are (global) correlation metrics:", metrics_correlation)

# Overall first and last frame in myeloid
minmax_frames_myel = [np.inf, -np.inf]
for c in myel_data.keys():
    for p in myel_data[c].keys():
        if myel_data[c][p]['f'].min() < minmax_frames_myel[0]:
            minmax_frames_myel[0] = myel_data[c][p]['f'].min()
        if myel_data[c][p]['f'].max() > minmax_frames_myel[1]:
            minmax_frames_myel[1] = myel_data[c][p]['f'].max()
print('\nOverall first & last frame in myel data:', minmax_frames_myel)

In [ ]:
### Flip shifts if desired (see note in `parameters` code cell above)

if flip_shifts:   
    
    # Flip parameters
    prof_min_shift  = -prof_min_shift  # Time point range of profile into past
    prof_max_shift  = -prof_max_shift  # Time point ange of profile into future
    
    # Flip data indexers
    for c in ecto_data.keys():
        for p in ecto_data[c].keys():
            
            # Flip similarities df columns
            flipped_cols = [
                c.split("-shift=")[0] + "-shift=" + str(-int(c.split("-shift=")[-1])) 
                if "-shift=" in c else c
                for c in myel_similarities[c][p].columns
            ]
            ecto_similarities[c][p].columns = flipped_cols
            myel_similarities[c][p].columns = flipped_cols
            
            # Flip profiles df columns
            for metric in ecto_profiles[c][p].keys():
                ecto_profiles[c][p][metric].columns = -ecto_profiles[c][p][metric].columns
                myel_profiles[c][p][metric].columns = -myel_profiles[c][p][metric].columns
                
            # Flip mean profiles series indices
            for metric in ecto_profs_mean[c][p].keys():
                ecto_profs_mean[c][p][metric].index = -ecto_profs_mean[c][p][metric].index
                myel_profs_mean[c][p][metric].index = -myel_profs_mean[c][p][metric].index

### Interactive visualizations compiled from processing pipeline

In [ ]:
### Visualize the data

@interact(show=False, condition=ecto_data.keys())
def show_by_condition(show=False, condition='norm'):
    
    if not show:
        return
    
    @interact(position=ecto_data[condition].keys(),
              ecto_fraction=['10%', '0%', '5%', '10%', '20%'],
              show_overlay=True)
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0],
                         ecto_fraction='10%',
                         show_overlay=True):
        
        # Select data
        ecto_df = ecto_data[condition][position]
        myel_df = myel_data[condition][position]
        
        # Handle overlay vs separate subplots
        if show_overlay:
            fig, ax = plt.subplots(1, figsize=(7, 7))
            ax = [ax, ax]
        else:
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        
        # Plot ectoderm tracks
        if ecto_fraction != '0%':
            for TID in ecto_df.index.unique()[::100//int(ecto_fraction[:-1])]:
                ax[0].scatter(
                    ecto_df.loc[ecto_df.index==TID, 'x'],
                    ecto_df.loc[ecto_df.index==TID, 'y'],
                    c=ecto_df.loc[ecto_df.index==TID, 'f'],
                    cmap='winter_r', s=5, alpha=0.3)
                
        # Plot myeloid tracks
        for TID in myel_df.index.unique():
            ax[1].scatter(
                myel_df.loc[myel_df.index==TID, 'x'],
                myel_df.loc[myel_df.index==TID, 'y'],
                c=myel_df.loc[myel_df.index==TID, 'f'],
                cmap='autumn_r', s=15, alpha=0.7)
        
        # Cosmetics
        for axis in ax:
            axis.axis('equal')
            axis.axis('off')
        
        # Finalize
        plt.tight_layout()

In [ ]:
### Visualize interpolated vectors

@interact(show=False, condition=ecto_data.keys())
def show_by_condition(show=False, condition='norm'):
    
    if not show:
        return
    
    @interact(position=ecto_data[condition].keys())
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        
        @interact(frame=(0, ecto_data[condition][position]['f'].max(), 1), 
                  scale=(0.01, 0.1, 0.01), quadrant=False)
        @savebutton
        def show_vectors(frame=ecto_data[condition][position]['f'].max()//2, 
                         scale=0.15, quadrant=False):
            
            # Select relevant data
            ecto_df = ecto_data[condition][position]
            myel_df = myel_data[condition][position]
            
            # Create frame masks
            ecto_fmask = (ecto_df['f'] == frame).values
            myel_fmask = (myel_df['f'] == frame).values
            myel_fmask_p1 = (myel_df['f'] == frame+1).values

            # Prep figure
            fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)

            # Plot observed vectors
            ax[0].quiver(ecto_df.loc[ecto_fmask, 'x'],  ecto_df.loc[ecto_fmask, 'y'],
                         ecto_df.loc[ecto_fmask, 'vx'], ecto_df.loc[ecto_fmask, 'vy'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='black', alpha=0.6, width=0.003)
            ax[1].quiver(myel_df.loc[myel_fmask, 'x'],  myel_df.loc[myel_fmask, 'y'],
                         myel_df.loc[myel_fmask, 'vx'], myel_df.loc[myel_fmask, 'vy'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='black', alpha=0.6, width=0.003)

            # Plot interpolated vectors
            ax[0].quiver(ecto_df.loc[ecto_fmask, 'x'], ecto_df.loc[ecto_fmask, 'y'],
                         ecto_df.loc[ecto_fmask, 'vx_itp'], ecto_df.loc[ecto_fmask, 'vy_itp'],
                         angles='xy', scale_units='xy', scale=scale,
                         color='red', alpha=0.6, width=0.003)
            ax[1].quiver(myel_df.loc[myel_fmask, 'x'], myel_df.loc[myel_fmask, 'y'],
                         myel_df.loc[myel_fmask, 'vx_itp'], myel_df.loc[myel_fmask, 'vy_itp'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='red', alpha=0.6, width=0.003)

            # Plot source points
            ax[0].scatter(ecto_df.loc[ecto_fmask, 'x'], ecto_df.loc[ecto_fmask, 'y'], 
                          c='darkblue', s=5, alpha=1.0)
            ax[1].scatter(myel_df.loc[myel_fmask, 'x'], myel_df.loc[myel_fmask, 'y'], 
                          c='darkblue', s=5, alpha=1.0)

            # Axis limits
            if quadrant:
                ax[0].set_xlim([0, 800])
                ax[0].set_ylim([700, 1500])

            # Labels
            ax[0].set_title(f'Ectoderm (t={frame*time_res}min)')
            ax[1].set_title(f'Myeloid (t={frame*time_res}min)')

            # Finish
            plt.tight_layout()

In [ ]:
### Visualize the resulting correlation/similarity time profiles

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys())
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        
        # Select relevant data
        ecto_profs = ecto_profiles[condition][position]
        myel_profs = myel_profiles[condition][position]
        
        # Prep
        fig, ax = plt.subplots(
            2, len(ecto_profs.keys()), 
            figsize=(1.1*len(ecto_profs.keys()), 8), 
            sharex=True, sharey=True)

        # For each metric...
        for m, metric in enumerate(ecto_profs.keys()):

            # Generate the heatmap
            ax[0,m].imshow(ecto_profs[metric], interpolation='none')
            ax[1,m].imshow(myel_profs[metric], interpolation='none')

            # Set cosmetics
            ax[0,m].set_title(metric, fontsize=9.5, rotation=45, ha="left")
            ax[1,m].set_xlabel('$shift$')
            if not flip_shifts:
                ax[1,m].set_xticks([0, (prof_max_shift-prof_min_shift)//2, prof_max_shift-prof_min_shift])
                ax[1,m].set_xticklabels([prof_min_shift, prof_min_shift+(prof_max_shift-prof_min_shift)//2, prof_max_shift])
            else:
                ax[1,m].set_xticks([0, (prof_min_shift-prof_max_shift)//2, prof_min_shift-prof_max_shift])
                ax[1,m].set_xticklabels([prof_min_shift, prof_min_shift+(prof_max_shift-prof_min_shift)//2, prof_max_shift])
            
        # Since indexers (not data) are flipped, imshow must be flipped as well!
        ax[0,m].invert_xaxis()

        # More cosmetics
        ax[0,0].set_ylabel('Ectoderm\n\n$frame$')
        ax[1,0].set_ylabel('Myeloid\n\n$frame$')

        # Finalize
        plt.tight_layout()

In [ ]:
### Visualize the resulting *averaged* correlation/similarity time profiles (per sample)

# Weird hack to prevent autoscrolling of widget output
style = """
    <style>
       .jupyter-widgets-output-area .output_scroll {
            height: unset !important;
            border-radius: unset !important;
            -webkit-box-shadow: unset !important;
            box-shadow: unset !important;
        }
        .jupyter-widgets-output-area  {
            height: auto !important;
        }
    </style>
    """
display(ipywidgets.HTML(style))

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    
    @interact(position=ecto_data[condition].keys(),
              yrange=["tight", "limits", "indiv"])
    @savebutton
    def show_by_position(
        position=list(ecto_data[condition].keys())[0], 
        yrange="tight"):
        
        # Select relevant data
        ecto_pms = ecto_profs_mean[condition][position]
        myel_pms = myel_profs_mean[condition][position]
        
        # Prep
        fig, ax = plt.subplots(
            int(np.ceil(len(ecto_pms.keys())/2)), 2, 
            figsize=(8, 2*int(np.ceil(len(ecto_pms.keys())/2))), 
            sharex=True, sharey=yrange=="common")
        if len(ecto_pms.keys()) % 2 != 0:
            ax[-1, -1].set_visible(False)

        # For each metric...
        for m, metric in enumerate(ecto_pms.keys()):
            
            # Plot profiles
            ax[m//2, m%2].plot(
                ecto_pms[metric].index * time_res, 
                ecto_pms[metric], 
                label='Ectoderm')
            ax[m//2, m%2].plot(
                myel_pms[metric].index * time_res, 
                myel_pms[metric], 
                label='Myeloid')

            # Set cosmetics
            ax[m//2, m%2].set_title(f"{condition} - {position} - {metric}")
            if (m//2) == 2:
                ax[m//2, m%2].set_xlabel('time shift [min]')
            if (m%2) == 0:
                ax[m//2, m%2].set_ylabel('metric')
                
            # Set y ranges
            if yrange == "tight":
                ax[m//2, m%2].set_ylim(yranges_tight[metric])
            if yrange == "limits":
                ax[m//2, m%2].set_ylim(yranges_limits[metric])
            
        # Crop x a bit to visualize peaks better
        plt.xlim(-100, 100)

        # Add midlines...
        for m in range(ax.size):
            ymin, ymax = ax[m//2, m%2].get_ylim()
            ax[m//2, m%2].vlines(0, ymin, ymax, color='k', lw=0.5, alpha=0.3, zorder=-1)
            ax[m//2, m%2].set_ylim(ymin, ymax)

        # Finalize
        plt.tight_layout()

### Remove problematic outliers identified as unhealthy explants

In [ ]:
### Outlier removal

# - Pos007 is the only sample that shows up as a substantial outlier across multiple different
#   analyses and visualizations, and has been indentified by Anh based on visual inspection to
#   most likely be an unhealthy explant or very poor fibronectin coating. It is therefore being
#   excluded from all further analysis.

#excluded_outliers = []
excluded_outliers = [("norm", "Pos007"), ]

for outlier in excluded_outliers:
    del ecto_data[outlier[0]][outlier[1]]
    del myel_data[outlier[0]][outlier[1]]
    del ecto_similarities[outlier[0]][outlier[1]]
    del myel_similarities[outlier[0]][outlier[1]]
    del ecto_profiles[outlier[0]][outlier[1]]
    del myel_profiles[outlier[0]][outlier[1]]
    del ecto_profs_mean[outlier[0]][outlier[1]]
    del myel_profs_mean[outlier[0]][outlier[1]]

### Analysis of correlation/similarity profiles over time shifts

In [ ]:
### Function to calculate profile stats (avg & std) across samples

def get_profile_stats(profs_mean, condition, metric):
    
    c = condition
    
    compiled = [profs_mean[c][p][metric] 
                for p in profs_mean[c].keys()]
    compiled = np.array(compiled)
    
    avg = pd.Series(
        np.mean(compiled, axis=0),
        index=profs_mean[c][list(profs_mean[c].keys())[0]][metric].index,
        name='< '+condition+' - '+metric+' - mean >')
    std = pd.Series(
        np.std(compiled, axis=0), 
        index=profs_mean[c][list(profs_mean[c].keys())[0]][metric].index,
        name='< '+condition+' - '+metric+' - std >')
    
    return avg, std

In [ ]:
### Function to visualize correlation/similarity profiles across samples

def show_profile(
    ecto_profs_mean, myel_profs_mean, metric, yrange="tight",
    show_samples=True, show_avg=True):
    
    # Prep
    fig, ax = plt.subplots(2, 1, figsize=(6, 5), sharex=True, sharey=yrange!="indiv")
    conditions = list(ecto_profs_mean.keys())
    ccycler = plt.rcParams["axes.prop_cycle"].by_key()['color']

    # Plot individual profiles
    if show_samples:
        for axi, c in enumerate(conditions):
            for p in ecto_profs_mean[c].keys():
                ax[axi].plot(
                    ecto_profs_mean[c][p][metric].index * time_res, 
                    ecto_profs_mean[c][p][metric], 
                    color=ccycler[0], lw=1, alpha=0.2)
                ax[axi].plot(
                    myel_profs_mean[c][p][metric].index * time_res, 
                    myel_profs_mean[c][p][metric], 
                    color=ccycler[1], lw=1, alpha=0.2)
        
    # Plot average profiles
    if show_avg:
        for axi, c in enumerate(conditions):
            ecto_avg, ecto_std = get_profile_stats(ecto_profs_mean, c, metric)
            myel_avg, myel_std = get_profile_stats(myel_profs_mean, c, metric)
            ax[axi].plot(
                ecto_avg.index * time_res, ecto_avg,
                color=ccycler[0], lw=2, alpha=0.8,
                label='Ectoderm' if c==conditions[0] else '_no_label_')
            ax[axi].plot(
                myel_avg.index * time_res, myel_avg,
                color=ccycler[1], lw=2, alpha=0.8,
                label='Myeloid' if c==conditions[0] else '_no_label_')
        
    # Set cosmetics
    ax[0].legend(fontsize=8, frameon=False)
    for axi, c in enumerate(conditions):
        ax[axi].set_title(f'{cond_dict[c]}', fontsize=10.5)
    ax[1].set_xlabel('time shift [min]')
    pub_ylbls = {
        "cos_sim" : "cosine similarity", "speed_corr" : "correlation of speeds",
        "traj_ortho_corr" : "correlation of\ntrajectory deviations"}
    ax[0].set_ylabel(pub_ylbls[metric] if metric in pub_ylbls else metric)
    ax[1].set_ylabel(pub_ylbls[metric] if metric in pub_ylbls else metric)
    
    # Axis settings
    plt.xlim(-100, 100)     # Crop x a bit to visualize peaks better
    if yrange == "tight":
        ax[0].set_ylim(yranges_tight[metric])
    if yrange == "limits":
        ax[0].set_ylim(yranges_limits[metric])

    # Add midlines...
    for axi in range(ax.size):
        ymin, ymax = ax[axi].get_ylim()
        ax[axi].vlines(0, ymin, ymax, color='k', lw=0.5, alpha=0.3, zorder=-1)
        ax[axi].set_ylim(ymin, ymax)

    # Finalize
    plt.tight_layout()

In [ ]:
### Show the correlation/similarity profiles

@interact(metric=metrics[2:], yrange=["tight", "limits", "indiv"])
@savebutton
def wrapper(metric="traj_dev_corr", yrange="tight"):
    show_profile(ecto_profs_mean, myel_profs_mean, metric, yrange)

In [ ]:
### Function to show correlations/similarities as boxplots

def show_prof_boxplot(
    ecto_profs_mean, myel_profs_mean,
    metric, group_labels, 
    shifts=[-1, 0, 1], yrange="tight", x_breaks=[]):

    # Prep
    fig, ax = plt.subplots(1, 4, figsize=(9, 4), sharey=yrange!="indiv")
    colors = {
        "ectoderm" : plt.rcParams["axes.prop_cycle"].by_key()['color'][0],
        "myeloid"  : plt.rcParams["axes.prop_cycle"].by_key()['color'][1]}

    # For each condition and cell type
    condXctype = itertools.product(ecto_profs_mean.keys(), colors.keys())
    for axis, (cond, ctype) in zip(ax, condXctype):
        
        # Select plot data
        plot_data = []
        plot_source = {
            "ectoderm" : ecto_profs_mean, "myeloid" : myel_profs_mean}[ctype]
        for shift in shifts:
            plot_data.append(
                [plot_source[cond][p][metric].loc[shift] 
                 for p in plot_source[cond].keys()
                 if not np.isnan(plot_source[cond][p][metric].loc[shift])])

        # Create boxplot
        bp = axis.boxplot(plot_data, widths=0.6, sym='', patch_artist=True)

        # Style boxplot
        for patch in bp['boxes']:
            patch.set(color=colors[ctype], alpha=0.5)
        for whisker in bp['whiskers']:
            whisker.set(color='black', linewidth=1.2, linestyle='-', alpha=0.5)
        for cap in bp['caps']:
            cap.set(linewidth=1.2, alpha=0.6)
        for median in bp['medians']:
            median.set(color='black', linewidth=1.2, alpha=0.5)

        # Add jittered data
        for i,p in enumerate(plot_data):
            y = p
            jitter_sigma = 0.10
            x = np.random.normal(i+1, jitter_sigma, size=len(y))
            x[x>(i+1+2*jitter_sigma)] = i+1+2*jitter_sigma
            x[x<(i+1-2*jitter_sigma)] = i+1-2*jitter_sigma
            axis.plot(x, y, '.', color=colors[ctype], markeredgecolor='k', alpha=0.7, ms=8)

        # Subplot cosmetics
        axis.set_xticklabels(np.array(shifts)*time_res, fontsize=8)
        axis.set_xlabel("shift [min]")
        axis.set_title(cond_dict[cond] + "\n" + ctype, fontsize=10.5)
        
    # Global cosmetics
    pub_ylbls = {
        "cos_sim" : "cosine similarity", "speed_corr" : "correlation of speeds",
        "traj_ortho_corr" : "correlation of trajectory deviations"}
    ax[0].set_ylabel(pub_ylbls[metric] if metric in pub_ylbls else metric)
    if yrange == "tight":
        ax[0].set_ylim(yranges_tight[metric])
    if yrange == "limits":
        ax[0].set_ylim(yranges_limits[metric])
        
    # Broken axes
    for break_x in x_breaks:
        for axis in ax:
            break_y, ymax = axis.get_ylim()
            axis.scatter(break_x, break_y, color='white', marker='s', s=80, clip_on=False, zorder=100)
            axis.text(break_x+0.050, break_y-0.002, r'//', fontsize=9, zorder=101,
                      horizontalalignment='center', verticalalignment='center')
            axis.set_ylim(break_y, ymax)  # (Somehow needed only for yrange="indiv")
    
    # Done
    plt.tight_layout()

In [ ]:
### Show the correlation/similarity boxplots; boxplots with shifts around zero

shifts = [-20, -2, -1, 0, 1, 2, 20]

@interact(metric=metrics[2:], yrange=["tight", "limits", "indiv"])
@savebutton
def wrapper(metric=metrics[2], yrange="tight"):
    
    group_labels = ['Ectoderm', 'Myeloid']
    show_prof_boxplot(
        ecto_profs_mean, myel_profs_mean, metric, group_labels, 
        shifts=shifts, yrange=yrange, x_breaks=[1.5, 6.5])

In [ ]:
### Get p-values for comparisons of interest

def get_profile_pval(cnd1, cnd2, ctp1, ctp2, sft1, sft2, metric, ptest="MWU"):
        
    # Get data
    data_sources = {"ectoderm" : ecto_profs_mean, "myeloid" : myel_profs_mean}
    ds1 = data_sources[ctp1]
    ds2 = data_sources[ctp2]
    d1 = [ds1[cnd1][p][metric].loc[sft1] for p in ds1[cnd1].keys()
          if not np.isnan(ds1[cnd1][p][metric].loc[sft1])]
    d2 = [ds2[cnd2][p][metric].loc[sft2] for p in ds2[cnd2].keys()
          if not np.isnan(ds2[cnd2][p][metric].loc[sft2])]
    
    # Compute p-value
    if ptest == "MWU":  # Unpaired
        stat, pval = stats.mannwhitneyu(d1, d2)
    elif ptest == "Wil":  # Paired
        stat, pval = stats.wilcoxon(d1, d2)
    
    return pval

# P-values between shift -30 and 0
shift0, shift1 = -20, {"ectoderm" : 0, "myeloid" : -1}
if flip_shifts:
    shift0, shift1 = -shift0, {k:-v for k,v in shift1.items()}
title = f"P-values between shift {shift0*time_res}min and the peak:"
print(title + "\n" + "-" * len(title))
for ctype in ["ectoderm", "myeloid"]:
    subtitle = f"{ctype} (peak shift = {shift1[ctype]*time_res}min)"
    print(f"\n  {subtitle}:" + "\n  " + "~" * (len(subtitle)+1))
    
    for cond in cond_dict:
        print(f"\n    {cond_dict[cond]}:")

        for metric in metrics_correlation[2:] + ["cos_sim"]:
            pval = get_profile_pval(
                cond, cond, ctype, ctype, shift0, shift1[ctype], metric, 
                #ptest="MWU",
                ptest="Wil",  # Swap to paired, since these comparisons *are* paired
            )
            print(f"      {metric:20} p={pval:.5f}")

# P-values between shift -1 and 0
shift0, shift1 = -1, 0
if flip_shifts:
    shift0, shift1 = -shift0, -shift1
title = f"P-values between shift {shift0*time_res}min and shift {shift1*time_res}min:"
print("\n")
print(title + "\n" + "-" * len(title))
for ctype in ["ectoderm", "myeloid"]:
    print(f"\n  {ctype}:" + "\n  " + "~" * (len(ctype)+1))
    
    for cond in cond_dict:
        print(f"\n    {cond_dict[cond]}:")

        for metric in metrics_correlation[2:] + ["cos_sim"]:
            pval = get_profile_pval(
                cond, cond, ctype, ctype, shift0, shift1, metric, 
                #ptest="MWU",
                ptest="Wil",  # Swap to paired, since these comparisons *are* paired
            )
            print(f"      {metric:20} p={pval:.5f}")

### Analysis of radial velocity profiles (comparison to active wetting model)

In [ ]:
### Prep: Create frame mask to exclude early and late frames from spatial plots

for c in myel_data.keys():
    for p in myel_data[c].keys():
        
        # Create profile mean range (pmr) mask
        with warnings.catch_warnings(action="ignore"):  # Ignore fragmentation perf warning...
            myel_data[c][p]["is_in_pmr"] = (myel_data[c][p]["f"] >= pmr[0]) & (myel_data[c][p]["f"] <= pmr[1])
            ecto_data[c][p]["is_in_pmr"] = (ecto_data[c][p]["f"] >= pmr[0]) & (ecto_data[c][p]["f"] <= pmr[1])
            myel_similarities[c][p]["is_in_pmr"] = myel_data[c][p]["is_in_pmr"].copy()
            ecto_similarities[c][p]["is_in_pmr"] = ecto_data[c][p]["is_in_pmr"].copy()
            
        # Perf: df defragmentation...
        myel_data[c][p] = myel_data[c][p].copy()
        ecto_data[c][p] = ecto_data[c][p].copy()
        myel_similarities[c][p] = myel_similarities[c][p].copy()
        ecto_similarities[c][p] = ecto_similarities[c][p].copy()

In [ ]:
### Function to show spatial distributions of data after radial projection

def show_spatial_radialbin(
    condition, cell_type, metric, shift,
    nbins=40, binrange=None, min_samples=5, 
    ax=None, figsize=(8, 3), color=None, label="_none_",
    return_data=False, mode="standard"):
    
    # Select cell type
    if cell_type == "myeloid":
        if metric in metrics_similarity:
            plot_data = myel_similarities[condition]
        else:
            plot_data = myel_data[condition]
            
    if cell_type == "ectoderm":
        if metric in metrics_similarity:
            plot_data = ecto_similarities[condition]
        else:
            plot_data = ecto_data[condition]
            
    # Add shift to metric
    if metric in metrics_similarity:
        metric += "-shift="+str(shift)
        
    # Get radial bins
    if binrange is None:
        binrange = [np.inf, -np.inf]
        for p in plot_data.keys():
            binrange[0] = min([binrange[0], plot_data[p]["r"].min()])
            binrange[1] = max([binrange[1], plot_data[p]["r"].max()])
    bins = np.linspace(binrange[0], binrange[1], nbins+1)
    bin_centers = bins[:-1] +  np.diff(bins) / 2.0
    
    # Compute average value per radial bin
    bin_values = np.full((len(plot_data), nbins), np.nan, float)
    for i, p in enumerate(plot_data.keys()):
        
        # Get relevant data
        p_df = plot_data[p][["r", metric, "is_in_pmr"]].copy()
        
        # Subselect profile mean range
        p_df = p_df.loc[p_df["is_in_pmr"], ["r", metric]]
        
        # Drop nans
        p_df = p_df.dropna()
        
        # Bin radial coordinate
        p_df["r_bins"] = pd.cut(p_df["r"], bins)
        
        # Get bin means
        bin_v = p_df.groupby("r_bins", observed=False)[metric].mean()
        
        # Find bins under minimum points limit
        bin_thresh = p_df.groupby("r_bins", observed=False).agg("size") >= min_samples
        
        # Store valid results
        bin_values[i, bin_thresh] = bin_v[bin_thresh]
        
    # Get overall mean
    with warnings.catch_warnings(category=RuntimeWarning, action="ignore"):
        bin_means = np.nanmean(
            bin_values, axis=0, 
            where=np.sum(~np.isnan(bin_values), axis=0) >= 3
        )
        
    # Prep plot
    if ax is None:
        plt.figure(figsize=figsize)
        ax = plt.gca()
    
    # Prep color
    if color is None:
        color = {
            "ectoderm" : "tab:blue",
            "myeloid"  : "tab:orange"
        }[cell_type]
    
    # Plot the samples
    for i,p in enumerate(plot_data.keys()):
        if mode == "standard":
            ax.plot(bin_centers, bin_values[i], alpha=0.35, lw=1.0, c=color)
        if mode == "revision":
            ax.scatter(
                bin_centers, bin_values[i], alpha=0.7, s=2.5, lw=0.3,
                edgecolors=color, facecolors="none", zorder=1/(i+1) + 1)
            
    # Plot the average
    if mode == "standard":
        ax.plot(bin_centers, bin_means, alpha=0.9, lw=2.0, c=color, label=label)
    if mode == "revision":
        ax.scatter(
            bin_centers, bin_means, alpha=1.0, s=8, marker="s", lw=0.5,
            edgecolors=color, facecolors="w", zorder=1/(i+1) + 3, label=label)
        
    # Return
    if not return_data:
        return ax
    else:
        return ax, bin_centers, bin_values, bin_means

In [ ]:
### Active wetting model of the ectoderm

# See `Active wetting model for ectoderm spreading` in the Materials and Methods for details!

# Prep
from scipy.special import iv
I = lambda x, n, m : iv(n, x)

# Ectoderm equation terms
v_cen_Jterm = lambda r, R, Ra, J : J/2.0 * (1.0 + (R**2.0/Ra**2.0)) * r
v_per_Jterm = lambda r, R,     J : J/2.0 * (1.0 + (R**2.0/r**2.0 )) * r
v_both_term = lambda r, R, T0, Lc, eta : T0*Lc/eta * (r * (I(R/Lc,0,20)/I(R/Lc,1,20) - Lc/R) - Lc * I(r/Lc,1,20)/I(R/Lc,1,20))
v_cen = lambda r, R, Ra, J, T0, Lc, eta: v_cen_Jterm(r, R, Ra, J) + v_both_term(r, R, T0, Lc, eta)
v_per = lambda r, R,     J, T0, Lc, eta: v_per_Jterm(r, R, J)     + v_both_term(r, R, T0, Lc, eta)

# Combined ectoderm equations
def v_ecto(r, R, Ra, J, T0, Lc, eta):
    v = np.empty_like(r, dtype=float)    
    v[r<=Ra] = v_cen(r[r<=Ra], R, Ra, J, T0, Lc, eta)
    v[r>Ra]  = v_per(r[r>Ra],  R,     J, T0, Lc, eta)
    v *= 60  # Time conversion from seconds to minutes to match myeloid model!
    return v

In [ ]:
### Saturating-cue model of the myeloid cells

pm = lambda r, R, Ra, J, T0, Lc, eta, h, a, vs : h/a*v_ecto(r, R, Ra, J, T0, Lc, eta) / (vs + v_ecto(r, R, Ra, J, T0, Lc, eta))
vm = lambda r, R, Ra, J, T0, Lc, eta, h, a, vs, v0 : v0 * pm(r, R, Ra, J, T0, Lc, eta, h, a, vs)

In [ ]:
### Combined model+data plot for ectoderm ray velocity

# Settings
condition = "norm"
cell_type = "ectoderm"
metric    = "vx_ray"
#mode = "standard"
mode = "revision"

# Plot the data
ax, bin_centers, bin_values, bin_means = show_spatial_radialbin(
    condition, cell_type, metric, shift=0,
    nbins=30, binrange=rpr, min_samples=5, 
    ax=None, figsize=(5, 3),
    color="darkblue", label="data",
    return_data=True, mode=mode,
)

# Superimpose model: set ectoderm parameters
R = 800      # To match data
Ra = 250     # To match data
#T0 =  18.0  # To recover non-influx model
T0 = 15.0    # Adjusted to match data
#J = 0.0     # To recover non-influx model
J = 1.6e-6   # Adjusted to match data
Lc, eta = 25.0, 50.0e6

# Superimpose model: add curve to the plot
plt.plot(
    np.arange(R),
    v_ecto(np.arange(R), R, Ra, J, T0, Lc, eta), 
    "-", c="royalblue", lw=1.5, zorder=-1,
    label="model",
)

# Indicate area of high-quality data
plt.axvspan(rpr[0], rpr[1], color='0.95', zorder=-10)

# Axis limits
ax.set_ylim(0.0, 0.50)
ax.set_xlim(0, ax.get_xlim()[-1])

# Legend
ax.legend()

# Labeling
ax.set_xlabel(r"Radial coordinate $r \ [\mu m]$")
ax.set_ylabel(r"Radial velocity $v_r \ [\mu m/min]$")
ax.set_title(f"condition: {condition} | cell type: {cell_type}")

## Save figure
#plt.savefig(
#    #r"..\Figures\radial_velocity_ectoderm_model-data.png", 
#    r"..\Figures\radial_velocity_ectoderm_model-data.pdf", 
#    transparent=True, dpi=300, bbox_inches="tight")

# Finalize
plt.show()

# Infer and report slope
regs = {"slope" : [], "r^2" : []}
for p, bv in zip(ecto_data[condition].keys(), bin_values):
    regression = reg = stats.linregress(bin_centers, bv)
    regs["slope"].append(reg.slope); regs["r^2"].append(reg.rvalue**2)
    print(f"{p} -- slope: {reg.slope:.6f}, intercept: {reg.intercept:.6f}, r-squared: {reg.rvalue**2:.2f}")
print(f"\nMean slope: {np.nanmean(regs["slope"]):.6f} um/min per um, mean r-squared: {np.nanmean(regs["r^2"]):.2f}")
print(f"Mean slope: {np.nanmean(regs["slope"])*60:.6f} um/h per um, mean r-squared: {np.nanmean(regs["r^2"]):.2f}")

# Also report model slope
model_reg = stats.linregress(np.arange(Ra, R), v_ecto(np.arange(Ra, R), R, Ra, J, T0, Lc, eta))
print(f"\nModel slope: {model_reg.slope:.6f} um/min per um, model r-squared: {model_reg.rvalue**2:.2f}")
print(f"Model slope: {model_reg.slope*60:.6f} um/h per um, model r-squared: {model_reg.rvalue**2:.2f}")

In [ ]:
### Combined model+data plot for ectoderm ray velocity: version binned by cluster size and zoomed

# Settings
condition = c = "norm"
metric    = "vx_ray"
extended_pmr = epmr = ( 60, 200)
extended_rpr = erpr = (250, 900)
nbins_r = 30
nbins_s = 4

# Find min and max values across dataset
min_r, max_r = np.inf, -np.inf
min_s, max_s = np.inf, -np.inf
for p in ecto_data[c]:
    is_in_pmr = (ecto_data[c][p]["f"] > epmr[0]) & (ecto_data[c][p]["f"] <= epmr[1])
    min_r = min([min_r, ecto_data[c][p]["r"].loc[is_in_pmr].min()])
    max_r = max([max_r, ecto_data[c][p]["r"].loc[is_in_pmr].max()])
    min_s = min([min_s, ecto_data[c][p]["clust_r"].loc[is_in_pmr].min()])
    max_s = max([max_s, ecto_data[c][p]["clust_r"].loc[is_in_pmr].max()])
    
# Clamp to extended radial profile range
min_r, max_r = max([min_r, erpr[0]]), min([max_r, erpr[1]])
min_s, max_s = max([min_s, erpr[0]]), min([max_s, erpr[1]])

# Clamp r to be within cluster size
max_r = min([max_r, max_s])

# Get the bins
bins_r = np.floor(np.linspace(min_r, max_r, nbins_r+1)).astype(int)
bins_s = np.floor(np.linspace(min_s, max_s, nbins_s+1)).astype(int)
#print(bins_r)
#print(bins_s)

# Prep plot
plt.figure(figsize=(5, 3))

# Set colors
colors = [plt.cm.winter_r(i/(nbins_s-1)) for i in range(nbins_s)]

# For each cluster size bin...
for bin_start, bin_end, color in zip(bins_s[:-1], bins_s[1:], colors):
    
    all_bin_means = []
    for p in ecto_data[c]:
    
        # Select data
        is_in_pmr = (ecto_data[c][p]["f"] > epmr[0]) & (ecto_data[c][p]["f"] <= epmr[1])
        p_df = ecto_data[c][p].loc[is_in_pmr, :].copy()

        # Mask by cluster size limit
        s_mask =  p_df["clust_r"] >  bin_start
        s_mask &= p_df["clust_r"] <= bin_end
        p_df = p_df.loc[s_mask, :].copy()

        # Mask radii down to max cluster size
        r_mask = p_df["r"] <= bin_end
        p_df = p_df.loc[r_mask, :]

        # Get mean per radius bin
        r_cut = pd.cut(
            p_df["r"], bins_r,
            labels=[int((bins_r[i]+bins_r[i+1])/2.0) for i in range(nbins_r)])
        bin_means = p_df.groupby(r_cut, observed=False)[metric].mean()

        # Plot mean outward velocity over radius
        plt.scatter(
            bin_means.index, bin_means, alpha=0.7, s=2.5, lw=0.3,
            edgecolors=[color for bm in bin_means], facecolors="none",
            zorder=1/bin_start + 1)
        
        # Keep result
        all_bin_means.append(bin_means)
        
    # Get overall bin means across samples
    with warnings.catch_warnings(category=RuntimeWarning, action="ignore"):
        overall_bin_means = np.nanmean(all_bin_means, axis=0)
    
    # Plot overall bin means
    plt.scatter(
        bin_means.index, overall_bin_means,
        alpha=1.0, s=8, marker="s", lw=0.5,
        edgecolors=[color for obm in overall_bin_means], facecolors="w",
        zorder=1/bin_start + 3,
        label=fr"R: {int(bin_start)}-{int(bin_end)} $\mu m$")

    # Superimpose Ricard's model
    bin_r = int((bin_end+bin_start)/2.0)
    plt.plot(
        np.arange(Ra, bin_r), v_per(np.arange(Ra, bin_r), bin_r, J, T0, Lc, eta) * 60, 
        "-", c=color, lw=1.3, zorder=1/bin_start + 2)

# Axis limits
plt.xlim(200, 930)
plt.ylim(0.00, 0.50)

# Legend
plt.legend(
    fontsize=8, frameon=False, 
    handletextpad=0.05, ncols=2, 
    columnspacing=0.5, loc=3)

# Labeling
plt.xlabel(r"Radial coordinate $r \ [\mu m]$")
plt.ylabel(r"Radial velocity $v_r \ [\mu m/min]$")
plt.title(f"Ectoderm $v$ profiles by cluster size")

## Save figure
#plt.savefig(
#    #r"..\Figures\radial_velocity_ectoderm_model-data_final.png", 
#    r"..\Figures\radial_velocity_ectoderm_model-data_final.pdf", 
#    transparent=True, dpi=300, bbox_inches="tight")

# Done
plt.show()

In [ ]:
### Combined model+data plot for myeloid ray velocity

# Settings
condition = "norm"
cell_type = "myeloid"
metric    = "vx_ray"
#mode = "standard"
mode = "revision"

# Plot the data
ax, bin_centers, bin_values, bin_means = show_spatial_radialbin(
    condition, cell_type, metric, shift=0,
    nbins=30, binrange=rpr, min_samples=5, 
    ax=None, figsize=(5, 3),
    color="orangered", label="data", 
    return_data=True, mode=mode,
)

# Superimpose model: set myeloid parameters
v0 = 1.0
a = 1.0
vs_fit = 0.01
#h = 0.95  # To recover original model
h = 0.85  # Adjusted to match data

# Superimpose model: add curve to the plot
plt.plot(
    np.arange(R),
    vm(np.arange(R), R, Ra, J, T0, Lc, eta, h, a, vs_fit, v0), 
    "-", c="darkorange", lw=1.5, zorder=-1,
    label="model",
)

# Indicate area of high-quality data
plt.axvspan(rpr[0], rpr[1], color='0.95', zorder=-10)

# Axis limits
ax.set_ylim(0.0, 1.75)
ax.set_xlim(0, ax.get_xlim()[-1])

# Legend
ax.legend()

# Labeling
ax.set_xlabel(r"Radial coordinate $r \ [\mu m]$")
ax.set_ylabel(r"Radial velocity $v_r \ [\mu m/min]$")
ax.set_title(f"Myeloid $v$ profile")

## Save figure
#plt.savefig(
#    #r"..\Figures\radial_velocity_myeloid_model-data.png",
#    r"..\Figures\radial_velocity_myeloid_model-data.pdf",
#    transparent=True, dpi=300, bbox_inches="tight")

# Finalize
plt.show()

In [ ]:
### Combined model+data plot for myeloid ray velocity: zoomed version

# Plot the data
ax, bin_centers, bin_values, bin_means = show_spatial_radialbin(
    condition, cell_type, metric, shift=0,
    nbins=30, binrange=rpr, min_samples=5, 
    ax=None, figsize=(5, 3),
    color="orangered", label="data", 
    return_data=True, mode=mode,
)

# Superimpose model: add curve to the plot
plt.plot(
    np.arange(240, 660), 
    vm(np.arange(240, 660), R, Ra, J, T0, Lc, eta, h, a, vs_fit, v0),
    "-", c="darkorange", lw=1.3, zorder=3,
)

# Axis limits
plt.xlim(230, 670)
plt.ylim(0.0, 1.75)

# Labeling
ax.set_xlabel(r"Radial coordinate $r \ [\mu m]$")
ax.set_ylabel(r"Radial velocity $v_r \ [\mu m/min]$")
ax.set_title(f"Myeloid $v$ profile")

# Extra cosmetics...
ax.set_xticks(np.arange(250, 651, 50))
ax.set_yticks([0.0, 0.5, 1.0, 1.5])

## Save figure
#plt.savefig(
#    #r"..\Figures\radial_velocity_myeloid_model-data_final.png",
#    r"..\Figures\radial_velocity_myeloid_model-data_final.pdf",
#    transparent=True, dpi=300, bbox_inches="tight")

# Finalize
plt.show()

### Plot ectoderm-myeloid transfer functions for cue saturation model

In [ ]:
### Subselect a clean dataset

myel_cue_df = {}

for c in myel_data:
    myel_cue_df[c] = {}
    
    for p in myel_data[c]:
        
        # Time range mask
        f_mask = (myel_data[c][p]["f"] > pmr[0]) & (myel_data[c][p]["f"] < pmr[1])
        
        # Radial range mask
        r_mask = (myel_data[c][p]["r"] > rpr[0]) & (myel_data[c][p]["r"] < rpr[1])
        
        # Pick relevant columns data
        columns = [
            "f", "t", "vy", "vx", "vy_n", "vx_n", "vy_itp", "vx_itp", 
            "v_mag", "v_itp_mag", "vx_ray", "vx_itp_ray", "vx_ray_n"
        ]
        
        # Select & copy data
        myel_cue_df[c][p] = myel_data[c][p].loc[f_mask & r_mask, columns].copy()
        
        # Add cosine distance...
        myel_cue_df[c][p]["cos_dist"] = 1.0 - myel_similarities[c][p].loc[f_mask & r_mask, "cos_sim-shift=0"]
        
        # Also add cosine similarity...
        myel_cue_df[c][p]["cos_sim"] = myel_similarities[c][p].loc[f_mask & r_mask, "cos_sim-shift=0"]

In [ ]:
### Compute relationship between local speeds

# Prep
c = "norm"
all_ecto = []
all_myel = []
xlims = (0.0, 0.5)
plt.figure(figsize=(5, 3))

# For each position/cluster...
for i,p in enumerate(myel_cue_df[c]):
    
    # Bin ectoderm velocity magnitude
    myel_cue_df[c][p]["ecto_bins"] = pd.cut(
        myel_cue_df[c][p]["v_itp_mag"], 
        bins=np.linspace(*xlims, 26)  # Standardized bin size
    )

    # Calculate binned averages for both
    binavg_ecto = myel_cue_df[c][p]["v_itp_mag"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=True).mean().values
    binavg_myel = myel_cue_df[c][p]["v_mag"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=True).mean().values
    
    # Plot the relationship
    plt.plot(
        binavg_ecto, binavg_myel, 
        c="tab:orange", alpha=0.35, lw=1.0,
        label="Samples" if i==0 else "_none_"
    )
    
    # Keep (relevant) data for fit
    xlims_mask = (binavg_ecto > xlims[0]) & (binavg_ecto < xlims[1])
    all_ecto.append(binavg_ecto[xlims_mask])
    all_myel.append(binavg_myel[xlims_mask])

# Add overall mean   
plt.plot(
    np.array(all_ecto).mean(axis=0), 
    np.array(all_myel).mean(axis=0), 
    c="tab:orange", lw=2.0, alpha=1.0,
    label="Mean"
)

# Legend
plt.legend(fontsize=9, frameon=False, loc=2)

# Axis limits
plt.xlim(*xlims)
plt.ylim(0.0, 3.5)

# Labels
plt.xlabel(
    r"Ectoderm flow speed $|\mathbf{v}_e|$ $[\mu m/min]$", 
    fontsize=10
)
plt.ylabel(
    r"Myeloid cell speed $|\mathbf{v}_m|$ $[\mu m/min]$", 
    fontsize=10
)

## Save for publication
#plt.title(" ")
#plt.savefig(
#    r"..\Figures\_Paper\ecto_speed_vs_myel_speed.pdf", 
#    transparent=True, bbox_inches="tight")

# Show figure
plt.show()

In [ ]:
### Cosine similiarity over local speed

# Prep
c = "norm"
all_ecto = []
all_myel = []
xlims = (0.0, 0.5)
plt.figure(figsize=(5, 3))

# For each position/cluster...
for i,p in enumerate(myel_cue_df[c]):
    
    # Bin ectoderm velocity magnitude
    myel_cue_df[c][p]["ecto_bins"] = pd.cut(
        myel_cue_df[c][p]["v_itp_mag"], 
        bins=np.linspace(*xlims, 26)  # Standardized bin size
    )

    # Calculate binned averages for both
    binavg_ecto = myel_cue_df[c][p]["v_itp_mag"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=True).mean().values
    binavg_myel = myel_cue_df[c][p]["cos_sim"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=True).mean().values
    
    # Plot the relationship
    plt.plot(
        binavg_ecto, binavg_myel, 
        c="tab:orange", alpha=0.35, lw=1.0,
        label="Samples" if i==0 else "_none_"
    )
    
    # Keep (relevant) data for fit
    xlims_mask = (binavg_ecto > xlims[0]) & (binavg_ecto < xlims[1])
    all_ecto.append(binavg_ecto[xlims_mask])
    all_myel.append(binavg_myel[xlims_mask])

# Add overall mean   
plt.plot(
    np.array(all_ecto).mean(axis=0), 
    np.array(all_myel).mean(axis=0), 
    c="tab:orange", lw=2.0, alpha=1.0,
    label="Mean"
)

# Legend
plt.legend(fontsize=9, frameon=False, loc=2)

# Axis limits
plt.xlim(*xlims)
plt.ylim(-0.5, 1.0)

# Labels
plt.xlabel(
    r"Ectoderm flow speed $|\mathbf{v}_e|$ $[\mu m/min]$", 
    fontsize=10
)
plt.ylabel(
    "Flow-alignment of myeloid cells\n(cosine similarity)", 
    fontsize=10
)

## Save for publication
#plt.title(" ")
#plt.savefig(
#    r"..\Figures\_Paper\ecto_speed_vs_cosine_similarity.pdf", 
#    transparent=True, bbox_inches="tight")

# Show figure
plt.show()